# TPOT API Documentation

**Tool:** TPOT (Tree-based Pipeline Optimization Tool)  
**Version:** 0.12.1  
**Purpose:** Automated machine learning pipeline optimization using genetic programming

This notebook demonstrates TPOT's core API and basic usage patterns.

In [1]:
'''
[BS12012025] AP_610_000001
[BS12012025] import all necessary modules
'''
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from tpot import TPOTClassifier, TPOTRegressor

## TPOT Core API

TPOT provides two main classes:
- `TPOTClassifier`: For classification tasks
- `TPOTRegressor`: For regression tasks

Both use genetic programming to search for optimal pipelines.

In [3]:
'''
[BS12042025] AP_610_000005
[BS12042025] Create synthetic dataset for a basic classification example
'''
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")

Training samples: 800
Test samples: 200
Features: 20


In [4]:
'''
[BS12042025] AP_610_000010
[BS12042025] Initialize TPOT classifier
'''
tpot_clf = TPOTClassifier(
    generations=3,           # Number of iterations
    population_size=10,      # Number of models per generation
    cv=3,                    # Cross-validation folds
    scoring='accuracy',      # Optimization metric
    verbosity=2,             # Print progress
    random_state=42,
    n_jobs=1                 # Parallel jobs
)

print("TPOT Configuration:")
print(f"  Generations: {tpot_clf.generations}")
print(f"  Population: {tpot_clf.population_size}")
print(f"  CV Folds: {tpot_clf.cv}")

TPOT Configuration:
  Generations: 3
  Population: 10
  CV Folds: 3


In [5]:
'''
[BS12042025] AP_610_000015
[BS12042025] Fit with TPOT
'''
print("\nOptimizing pipeline...")
tpot_clf.fit(X_train, y_train)


Optimizing pipeline...


Version 0.12.1 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


                                                                                
Generation 1 - Current best internal CV score: 0.8912590464926361
                                                                                
Generation 2 - Current best internal CV score: 0.9037669079064702
                                                                                
Generation 3 - Current best internal CV score: 0.9174903551012363
                                                                                
Best pipeline: RandomForestClassifier(MLPClassifier(input_matrix, alpha=0.001, learning_rate_init=0.1), bootstrap=True, criterion=gini, max_features=0.2, min_samples_leaf=8, min_samples_split=4, n_estimators=100)


TPOTClassifier(cv=3, generations=3, population_size=10, random_state=42,
               scoring='accuracy', verbosity=2)

In [6]:
'''
[BS12042025] AP_610_000020
[BS12042025] Evaluate results
'''
y_pred = tpot_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.2%}")
print(f"\nBest Pipeline:")
print(tpot_clf.fitted_pipeline_)


Test Accuracy: 94.00%

Best Pipeline:
Pipeline(steps=[('stackingestimator',
                 StackingEstimator(estimator=MLPClassifier(alpha=0.001,
                                                           learning_rate_init=0.1,
                                                           random_state=42))),
                ('randomforestclassifier',
                 RandomForestClassifier(max_features=0.2, min_samples_leaf=8,
                                        min_samples_split=4,
                                        random_state=42))])


In [7]:
'''
[BS12042025] AP_610_000025
[BS12042025] export pipeline
'''
tpot_clf.export('api_tpot_best_pipeline.py')
print("\n✅ Exported optimized pipeline to api_tpot_best_pipeline.py")


✅ Exported optimized pipeline to api_tpot_best_pipeline.py


## TPOT Configuration Options

### Key Parameters

**Search Parameters:**
- `generations`: Number of iterations (higher = more optimization)
- `population_size`: Models per generation (higher = broader search)
- `offspring_size`: New models per generation

**Validation:**
- `cv`: Cross-validation folds or strategy
- `scoring`: Metric to optimize ('accuracy', 'f1', 'roc_auc', etc.)

**Computational:**
- `n_jobs`: Parallel workers (-1 = all cores)
- `max_time_mins`: Time limit for optimization
- `max_eval_time_mins`: Time limit per model

**Model Selection:**
- `config_dict`: Limit to specific model types
- Available: 'TPOT light', 'TPOT MDR', 'TPOT sparse', or custom

In [8]:
'''
[BS12042025] AP_610_000030
[BS12042025] Create a custom configuration by limiting to specific models
'''
custom_config = {
    'sklearn.ensemble.RandomForestClassifier': {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5]
    },
    'sklearn.linear_model.LogisticRegression': {
        'C': [0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2']
    }
}

tpot_custom = TPOTClassifier(
    generations=2,
    population_size=5,
    config_dict=custom_config,
    verbosity=2,
    random_state=42
)

print("Custom TPOT with limited model space:")
print("  Models: RandomForest, LogisticRegression only")

Custom TPOT with limited model space:
  Models: RandomForest, LogisticRegression only


In [9]:
'''
[BS12042025] AP_610_000035
[BS12042025] Create a regression example
'''
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=10,
    noise=10,
    random_state=42
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

tpot_reg = TPOTRegressor(
    generations=3,
    population_size=10,
    scoring='neg_mean_squared_error',
    verbosity=2,
    random_state=42
)

print("\nOptimizing regression pipeline...")
tpot_reg.fit(X_train_r, y_train_r)

y_pred_r = tpot_reg.predict(X_test_r)
mse = mean_squared_error(y_test_r, y_pred_r)
print(f"\nTest MSE: {mse:.2f}")


Optimizing regression pipeline...


Version 0.12.1 of tpot is outdated. Version 1.1.0 was released Thursday July 03, 2025.


                                                                                
Generation 1 - Current best internal CV score: -95.58462748062524
                                                                                
Generation 2 - Current best internal CV score: -95.58462748062524
                                                                                
Generation 3 - Current best internal CV score: -95.58462748062509
                                                                                
Best pipeline: LassoLarsCV(input_matrix, normalize=False)

Test MSE: 97.10


## Key Concepts

### Genetic Programming
TPOT uses evolutionary algorithms to optimize pipelines:
1. **Initialize** population of random pipelines
2. **Evaluate** each pipeline using cross-validation
3. **Select** best performers
4. **Breed** new pipelines through crossover and mutation
5. **Repeat** for specified generations

### Pipeline Components
TPOT searches over:
- Feature preprocessing (scaling, selection, engineering)
- Model selection (classifiers/regressors)
- Hyperparameters for each component

### Output
- `fitted_pipeline_`: Best scikit-learn pipeline found
- `export()`: Save pipeline as Python code
- Standard scikit-learn interface (fit/predict)

## Best Practices

1. **Start small**: Use few generations/population for initial testing
2. **Use time limits**: `max_time_mins` prevents endless runs
3. **Enable verbosity**: Track progress with `verbosity=2`
4. **Save checkpoints**: Use `periodic_checkpoint_folder`
5. **Limit search space**: Use `config_dict` for faster convergence
6. **Proper validation**: Use appropriate CV strategy for your data
7. **Export pipelines**: Always save the best pipeline with `export()`

## Common Issues

- **Memory**: Reduce `population_size` or use `subsample`
- **Slow**: Reduce `generations` or set `max_time_mins`
- **Overfitting**: Use proper CV and avoid small datasets
- **Stuck**: Try different `config_dict` or `random_state`